# Statistics — McNemar significance tests and bootstrap confidence intervals

This notebook reproduces every statistical result reported in the manuscript and supplementary
materials that is computed *after* a model-training notebook finishes — McNemar's exact test
(with Holm-Bonferroni correction across the nine seed-length combinations) for the fragmentation
sweep (Table 5 / Supplementary Table S12) and the PR+RT amplicon comparison (Supplementary Table
S8/S9), and a percentile bootstrap 95% CI for the full-length macro-F1 (Supplementary Table S10).

**Inputs.** The per-sequence prediction CSVs saved by the ResNet-18 no-GaussianNoise rerun
(`../02_fragmentation_sweep/results/resnet18_A_fragmentation_predictions.csv` and
`../06_pr_rt_amplicon/results/resnet18_D_amplicon_predictions.csv`) — see `../../results/`.

**Method.** `statsmodels.stats.contingency_tables.mcnemar(table, exact=True)` for the raw p-value;
Holm-Bonferroni step-down correction (`p_holm = min(1.0, p_raw * (m - rank + 1))`, cumulative max)
across the 9 (fragmentation) or 3 (amplicon, per condition) comparisons tested in each family.
Bootstrap: 2,000 resamples, `numpy.random.default_rng(42)`, percentile method.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import f1_score

FRAG_PRED = "../../results/02_fragmentation_sweep/resnet18_A_fragmentation_predictions.csv"
AMP_PRED  = "../../results/06_pr_rt_amplicon/resnet18_D_amplicon_predictions.csv"


## 1. Fragmentation sweep — SVM vs. ResNet-18, McNemar + Holm-Bonferroni (Table 5 / S12)

In [ ]:
def holm_bonferroni(pvals):
    order = np.argsort(pvals)
    m = len(pvals)
    adj = np.empty(m)
    running_max = 0.0
    for rank, idx in enumerate(order):
        corrected = min(1.0, pvals[idx] * (m - rank))
        running_max = max(running_max, corrected)
        adj[idx] = running_max
    return adj

df = pd.read_csv(FRAG_PRED)
rows = []
for (length, seed), g in df.groupby(["length", "seed"]):
    svm_correct = (g["y_pred_svm"] == g["y_true"])
    r18_correct = (g["y_pred_resnet18"] == g["y_true"])
    svm_only = int((svm_correct & ~r18_correct).sum())
    r18_only = int((~svm_correct & r18_correct).sum())
    table = [[0, svm_only], [r18_only, 0]]
    p_raw = mcnemar(table, exact=True).pvalue
    rows.append(dict(length=length, seed=seed, n=len(g), svm_only_correct=svm_only,
                      resnet18_only_correct=r18_only, p_raw=p_raw))

res = pd.DataFrame(rows)
length_order = {"300": 0, "500": 1, "1000": 2}
res = res[res["length"].isin(length_order)].sort_values(["length", "seed"],
         key=lambda s: s.map(length_order) if s.name == "length" else s)
res["p_holm"] = holm_bonferroni(res["p_raw"].values)
res["significant"] = res["p_holm"] < 0.05
res.to_csv("../../results/08_statistics/mcnemar_fragmentation_resnet18_nonoise.csv", index=False)
print(res.to_string(index=False))
print(f"\n{res['significant'].sum()} of {len(res)} seed-length combinations significant after Holm-Bonferroni correction.")

## 2. PR+RT amplicon — SVM vs. ResNet-18, McNemar + Holm-Bonferroni, intact and dropout (Table S8/S9)

In [ ]:
df_amp = pd.read_csv(AMP_PRED)
rows = []
for (inp, seed), g in df_amp.groupby(["input", "seed"]):
    svm_correct = (g["y_pred_svm"] == g["y_true"])
    r18_correct = (g["y_pred_resnet18"] == g["y_true"])
    svm_only = int((svm_correct & ~r18_correct).sum())
    r18_only = int((~svm_correct & r18_correct).sum())
    table = [[0, svm_only], [r18_only, 0]]
    p_raw = mcnemar(table, exact=True).pvalue
    rows.append(dict(input=inp, seed=seed, n=len(g), svm_only_correct=svm_only,
                      resnet18_only_correct=r18_only, p_raw=p_raw))

res_amp = pd.DataFrame(rows)
res_amp["p_holm"] = res_amp.groupby("input")["p_raw"].transform(
    lambda s: holm_bonferroni(s.values))
res_amp["significant"] = res_amp["p_holm"] < 0.05
res_amp.to_csv("../../results/08_statistics/mcnemar_amplicon_resnet18_nonoise.csv", index=False)
print(res_amp.to_string(index=False))

## 3. Bootstrap 95% CI — full-length macro-F1, all four models (Table S10)

In [ ]:
def bootstrap_f1_ci(y_true, y_pred, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    boot_f1 = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_f1[b] = f1_score(y_true[idx], y_pred[idx], average="macro")
    point = f1_score(y_true, y_pred, average="macro")
    lo, hi = np.percentile(boot_f1, [2.5, 97.5])
    return dict(point=round(point, 4), ci_lower=round(lo, 4), ci_upper=round(hi, 4))

df_full = df[df["length"] == "full"]
y_true = df_full["y_true"].values
ci_svm = bootstrap_f1_ci(y_true, df_full["y_pred_svm"].values)
ci_r18 = bootstrap_f1_ci(y_true, df_full["y_pred_resnet18"].values)
print("SVM (full length, pooled 3 seeds):     ", ci_svm)
print("ResNet-18 (full length, pooled 3 seeds):", ci_r18)
pd.DataFrame([dict(model="svm", **ci_svm), dict(model="resnet18_nonoise", **ci_r18)]).to_csv(
    "../../results/08_statistics/bootstrap_ci_fulllength.csv", index=False)

## 4. MinHash-grouped vs. CD-HIT-97% split — SVM and Random Forest (Supplementary Table S14 input)

Reruns the linear SVM and Random Forest under the dataset's built-in MinHash near-duplicate
`cluster` column (instead of CD-HIT-97%) at full length, 3 seeds — CPU-only, a few minutes total.
Pairs against the CD-HIT-97% values already reported in Table 3 / Supplementary Table S4.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score

prepared = pd.read_csv("../../data/prepared_sequences_clean.csv.gz")
le = LabelEncoder(); y_all = le.fit_transform(prepared["subtype"].values)
X_all = prepared["sequence"].values
grp_minhash = prepared["cluster"].values

def kmers_str(seq, k=6):
    return " ".join(seq[i:i+k] for i in range(len(seq) - k + 1))

def split_grouped(X, y, grp, seed):
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(X[trv], y[trv], grp[trv]))
    return trv[tr_rel], trv[va_rel], te

rows = []
for seed in [0, 1, 2]:
    tr, va, te = split_grouped(X_all, y_all, grp_minhash, seed)
    ytr, yte = y_all[tr], y_all[te]
    Ktr = [kmers_str(s) for s in X_all[tr]]
    Kte = [kmers_str(s) for s in X_all[te]]
    vec = TfidfVectorizer(max_features=512); vec.fit(Ktr)
    Ttr = vec.transform(Ktr).toarray().astype(np.float32)
    Tte = vec.transform(Kte).toarray().astype(np.float32)

    svm = LinearSVC(class_weight="balanced", random_state=seed, max_iter=20000).fit(Ttr, ytr)
    pred_svm = svm.predict(Tte)
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=seed,
                                 n_jobs=-1).fit(Ttr, ytr)
    pred_rf = rf.predict(Tte)

    rows.append(dict(seed=seed, model="svm", f1=round(f1_score(yte, pred_svm, average="macro"), 4),
                      bal_acc=round(balanced_accuracy_score(yte, pred_svm), 4)))
    rows.append(dict(seed=seed, model="rf", f1=round(f1_score(yte, pred_rf, average="macro"), 4),
                      bal_acc=round(balanced_accuracy_score(yte, pred_rf), 4)))
    print(f"seed {seed} done")

out = pd.DataFrame(rows)
out.to_csv("../../results/03_split_ablations/svm_rf_minhash_fulllength_raw.csv", index=False)
print(out.groupby("model")[["f1", "bal_acc"]].agg(["mean", "std"]))